<div style="background:#0d1117;padding:32px 28px 24px;border-radius:12px;margin-bottom:8px">
<h1 style="color:#e6edf3;font-size:2rem;margin:0 0 6px">🛫 Revenue Management</h1>
<h2 style="color:#9da5b4;font-size:1.1rem;font-weight:400;margin:0 0 18px">A matemática por trás da passagem aérea</h2>
<p style="color:#9da5b4;font-size:.95rem;line-height:1.7;margin:0">
Uma companhia aérea não tenta simplesmente <strong style='color:#e6edf3'>lotar o avião</strong>.
O objetivo é <strong style='color:#06d6a0'>maximizar a receita</strong> de uma capacidade limitada.<br>
A pergunta central é: <em style='color:#ffd166'>"É melhor vender um assento barato agora ou reservar esse espaço
para alguém disposto a pagar muito mais depois?"</em>
</p>
</div>

---

**Estrutura do notebook:**

| # | Seção | O que veremos |
|---|-------|---------------|
| 1 | Custos do voo | Por que o custo médio não determina o preço |
| 2 | Preço único | Nenhum preço único fecha a conta |
| 3 | Discriminação de preços | Como as 3 classes tarifárias funcionam |
| 4 | EMSR-b | O algoritmo de proteção de assentos |
| 5 | Cenário determinístico | Ocupação máxima vs. Revenue Management |
| 6 | Monte Carlo | 5.000 voos com demanda incerta |
| 7 | A passagem de R$ 99 | O que ela realmente custa à empresa |
| 8 | Contexto histórico | American Airlines, DINAMO e People Express |

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from scipy.stats import norm

np.random.seed(42)

# ── Estética dark ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#161b22',
    'savefig.facecolor': '#0d1117',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#c9d1d9',
    'text.color'       : '#e6edf3',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'grid.color'       : '#21262d',
    'grid.linewidth'   : 0.6,
    'axes.grid'        : True,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'figure.dpi'       : 110,
    'savefig.dpi'      : 200,
    'figure.autolayout': True,
})

# Paleta de cores
C = {
    'naive' : '#ef476f',   # estratégia ingênua  — vermelho-rosa
    'rm'    : '#06d6a0',   # revenue management  — verde-teal
    'promo' : '#118ab2',   # classe promocional  — azul
    'inter' : '#ffd166',   # classe intermediária — âmbar
    'ultra' : '#ef476f',   # última hora          — vermelho
    'neutro': '#8b949e',   # linhas de referência
}

def brl(v):
    """Formata número como moeda BRL."""
    s = f'{abs(v):,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'-R$ {s}' if v < 0 else f'R$ {s}'

def titulo_secao(n, texto):
    sep = '─' * 68
    print(f'\n{sep}')
    print(f'  {n}. {texto}')
    print(sep)

print('✅  Setup concluído.')

---
## 1 · O problema econômico — custos de um voo doméstico

Um avião possui custos altíssimos **antes mesmo de transportar qualquer passageiro**.
Chamamos de **custo fixo** tudo que incorre independentemente do número de passageiros:
leasing da aeronave, tripulação, manutenção, taxas aeroportuárias e combustível de base.

O **custo marginal** (ou custo variável por passageiro) cobre itens como catering,
bagagem e taxas de embarque — bem menor do que o custo fixo.

> **Implicação direta:** depois que o avião está programado para voar,
> colocar mais um passageiro a bordo tem custo adicional pequeno.
> Isso cria o espaço para vender alguns assentos a preços baixos **sem prejudicar a operação**
> — desde que os outros assentos paguem o custo fixo.

In [ ]:
titulo_secao(1, 'Estrutura de custos — Airbus A320 / Boeing 737 (2h30 de voo)')

CAPACIDADE     = 180      # assentos
CUSTO_FIXO     = 92_000   # R$ — operação + combustível + taxas (conforme enunciado)
CUSTO_MARGINAL = 50       # R$ — por passageiro

itens_custo = {
    'Operação, tripulação e manutenção (R$25k/h × 2,5h)': 62_500,
    'Combustível, taxas aeroportuárias e variáveis'      : 29_500,
}

print()
for descricao, valor in itens_custo.items():
    print(f'  {descricao:<52}  {brl(valor):>12}')
print(f'  {"CUSTO FIXO TOTAL DO VOO":<52}  {brl(CUSTO_FIXO):>12}')
print()
print(f'  Custo médio por assento (avião lotado)           {brl(CUSTO_FIXO / CAPACIDADE):>12}')
print(f'  Custo marginal por passageiro                    {brl(CUSTO_MARGINAL):>12}')
print(f'  Custo total por passageiro (avião lotado)        {brl(CUSTO_FIXO/CAPACIDADE + CUSTO_MARGINAL):>12}')
print()

# Gráfico de pizza dos custos
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
valores  = list(itens_custo.values())
rotulos  = ['Operação,\ntripulação e\nmanutenção', 'Combustível,\ntaxas e\nvariáveis']
cores_pi = ['#118ab2', '#06d6a0']
wedges, texts, autotexts = ax.pie(
    valores, labels=rotulos, colors=cores_pi, autopct='%1.0f%%',
    startangle=90, pctdistance=0.65,
    textprops={'fontsize': 9, 'color': '#e6edf3'},
    wedgeprops={'linewidth': 1.5, 'edgecolor': '#0d1117'},
)
for at in autotexts:
    at.set_color('#0d1117'); at.set_fontweight('bold')
ax.set_title(f'Composição do custo fixo\n({brl(CUSTO_FIXO)} por voo)', weight='bold', pad=10)

# Custo por assento conforme ocupação
ax2 = axes[1]
ocups = np.arange(60, 181)
custo_medio = CUSTO_FIXO / ocups
ax2.plot(ocups, custo_medio, color=C['naive'], lw=2.5)
ax2.axhline(CUSTO_MARGINAL, color=C['rm'], lw=1.5, ls='--', label=f'Custo marginal ({brl(CUSTO_MARGINAL)})')
ax2.axvline(CAPACIDADE, color=C['neutro'], lw=1, ls=':', label=f'Capacidade ({CAPACIDADE} pax)')
ax2.set_xlabel('Passageiros a bordo')
ax2.set_ylabel('Custo fixo por passageiro (R$)')
ax2.set_title('Custo médio cai com a ocupação\n(economia de escala no voo)', weight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax2.legend(fontsize=9, facecolor='#161b22', edgecolor='#30363d')
ax2.set_facecolor('#161b22')

fig.suptitle('1 · Estrutura de Custos do Voo', fontsize=13, weight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 2 · O erro do preço único

Antes de entender o Revenue Management, vamos testar a alternativa mais simples:
**cobrar o mesmo preço de todo mundo**.

Para cada preço candidato, calculamos quantos passageiros estão dispostos a pagar aquele valor
(demanda acumulada das classes com preço ≥ candidato), respeitando a capacidade do avião.

In [ ]:
titulo_secao(2, 'Simulação de preço único — nenhuma opção fecha a conta')

# Definição das classes tarifárias (usadas em todo o notebook)
classes = pd.DataFrame({
    'classe'        : ['Promocional', 'Intermediária', 'Última hora'],
    'preco'         : [200, 500, 2_000],
    'demanda_media' : [140, 80, 35],
    'demanda_dp'    : [30, 18, 10],
})

def demanda_preco_unico(p):
    """Passageiros dispostos a pagar >= p (limitado pela capacidade)."""
    return min(int(classes.loc[classes['preco'] >= p, 'demanda_media'].sum()), CAPACIDADE)

candidatos = [99, 200, 500, 2_000]
rows = []
for p in candidatos:
    q    = demanda_preco_unico(p)
    rec  = p * q
    cst  = CUSTO_FIXO + CUSTO_MARGINAL * q
    lucro = rec - cst
    rows.append({'Preço único': f'R$ {p:,.0f}'.replace(',','.'),
                 'Pax'        : q,
                 'Ocupação'   : f'{100*q/CAPACIDADE:.0f}%',
                 'Receita'    : brl(rec),
                 'Custo'      : brl(cst),
                 'Lucro'      : brl(lucro),
                 '_lucro'     : lucro})

df_pu = pd.DataFrame(rows)
print()
print(df_pu.drop(columns='_lucro').to_string(index=False))
print()
print('>>> Nenhum preço único gera lucro neste voo!')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels_pu   = [r['Preço único'] for r in rows]
receitas_pu = [r['_lucro'] + (CUSTO_FIXO + CUSTO_MARGINAL * r['Pax']) for r in rows]
lucros_pu   = [r['_lucro'] for r in rows]
cores_lucro = [C['rm'] if l > 0 else C['naive'] for l in lucros_pu]

custo_ref = CUSTO_FIXO + CUSTO_MARGINAL * CAPACIDADE   # custo com avião lotado

ax = axes[0]
bars = ax.bar(labels_pu, receitas_pu, color=[C['promo'], C['inter'], C['rm'], C['naive']], alpha=0.85)
ax.axhline(custo_ref, color='white', ls='--', lw=1.4, label=f'Custo (lotado): {brl(custo_ref)}')
ax.set_ylabel('Receita (R$)')
ax.set_title('Receita por preço único\n(linha = custo total com avião lotado)', weight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.legend(fontsize=8, facecolor='#161b22', edgecolor='#30363d')
for b, v in zip(bars, receitas_pu):
    ax.text(b.get_x()+b.get_width()/2, v+800, brl(v), ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
bars2 = ax2.bar(labels_pu, lucros_pu, color=cores_lucro, alpha=0.85)
ax2.axhline(0, color='white', lw=1.5)
ax2.set_ylabel('Lucro (R$)')
ax2.set_title('Lucro por preço único\n(vermelho = prejuízo)', weight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
for b, v in zip(bars2, lucros_pu):
    yp = v+500 if v >= 0 else v-2000
    va = 'bottom' if v >= 0 else 'top'
    ax2.text(b.get_x()+b.get_width()/2, yp, brl(v), ha='center', va=va, fontsize=8)

fig.suptitle('2 · Nenhum preço único resolve — a solução é a segmentação', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

---
## 3 · A solução: discriminação de preços (classes tarifárias)

O Revenue Management divide os passageiros em **classes tarifárias** com base na disposição a pagar
e no momento da compra:

| Classe | Preço | Perfil típico | Compra com |
|--------|-------|---------------|------------|
| Promocional | R\$ 200 | Turista, sensível a preço | 4–8 semanas de antecedência |
| Intermediária | R\$ 500 | Viagem planejada | 2–4 semanas |
| Última hora | R\$ 2.000 | Negócios / urgência | < 7 dias |

A demanda de cada classe é modelada como uma **distribuição normal** com média e desvio-padrão
estimados a partir do histórico de voos similares.

In [ ]:
titulo_secao(3, 'Distribuição de demanda por classe tarifária')

print()
print(classes.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4.5))

cores_cl = [C['promo'], C['inter'], C['ultra']]
x = np.linspace(0, 200, 400)
for (_, row), cor in zip(classes.iterrows(), cores_cl):
    y = norm.pdf(x, row['demanda_media'], row['demanda_dp'])
    ax.plot(x, y, color=cor, lw=2.5, label=f"{row['classe']} (R\$ {row['preco']:,}) — média {row['demanda_media']} pax")
    ax.fill_between(x, y, alpha=0.15, color=cor)
    ax.axvline(row['demanda_media'], color=cor, lw=1, ls=':')

ax.set_xlabel('Número de passageiros que solicitam esta classe')
ax.set_ylabel('Densidade de probabilidade')
ax.set_title('3 · Incerteza na demanda — cada voo tem demanda diferente por classe', weight='bold')
ax.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)
ax.set_facecolor('#161b22')
plt.tight_layout()
plt.show()

---
## 4 · O algoritmo EMSR-b (Belobaba, 1987–1989)

A ideia central não é definir um preço — é decidir **quantos assentos proteger** para as classes
mais caras, mesmo que isso signifique recusar uma venda barata garantida agora.

**Regra de Littlewood (1972):** proteja um assento para a tarifa alta `p_h` enquanto a probabilidade
de vender esse assento satisfazer `P(D_h ≥ 1) · p_h > p_l` (tarifa baixa atual).

**EMSR-b (Expected Marginal Seat Revenue-b):**  
Para cada classe `j` (da mais barata para a mais cara), o nível de proteção `y_j` é o fractil
da demanda agregada das classes superiores onde o preço de `j` "empata" com a receita marginal
esperada de guardar um assento:

$$y_j = F^{-1}\!\left(1 - \frac{p_j}{\bar{p}_{>j}}\;,\; \mu_{>j},\, \sigma_{>j}\right)$$

Onde $\bar{p}_{>j}$ é o preço médio ponderado das classes acima de `j` e $F^{-1}$ é a
inversa da distribuição normal acumulada da demanda agregada.

In [ ]:
titulo_secao(4, 'Algoritmo EMSR-b — cálculo dos níveis de proteção')

def emsr_b(precos, medias, dps):
    """
    EMSR-b de Belobaba (1987/1989), extensão da regra de Littlewood (1972).
    Parâmetros ordenados da classe mais BARATA para a mais CARA.
    Retorna lista de níveis de proteção y_j (um por classe, exceto a última).
    """
    n          = len(precos)
    protecoes  = []
    for j in range(n - 1):
        # Agrega demanda das classes superiores
        p_acima  = precos[j+1:]
        m_acima  = medias[j+1:]
        d_acima  = dps[j+1:]
        mu       = sum(m_acima)
        sigma    = np.sqrt(sum(d**2 for d in d_acima))
        p_medio  = sum(p * m for p, m in zip(p_acima, m_acima)) / mu

        # Fractil crítico e nível de proteção
        fractil = float(np.clip(1.0 - precos[j] / p_medio, 1e-9, 1 - 1e-9))
        y_j     = norm.ppf(fractil, loc=mu, scale=sigma)
        protecoes.append(max(0.0, y_j))
    return protecoes


precos_l = classes['preco'].tolist()
medias_l = classes['demanda_media'].tolist()
dps_l    = classes['demanda_dp'].tolist()

protecoes = emsr_b(precos_l, medias_l, dps_l)

print()
print('  Classe tarifária    │  Proteger para classes acima  │  Booking limit')
print('  ──────────────────────────────────────────────────────────────────────')
restante = CAPACIDADE
booking_limits = []
for i, (cls, prot) in enumerate(zip(classes['classe'][:-1], protecoes)):
    limite = max(0, restante - int(round(prot)))
    booking_limits.append(limite)
    print(f'  {cls:<20} │  {prot:>6.1f} assentos           │  {limite} de {CAPACIDADE}')
booking_limits.append(restante)  # última classe: sem limite acima
print(f'  {"Última hora":<20} │  —  (classe topo)            │  sem restrição')
print()
print(f'  >>> Máximo de {booking_limits[0]} passageiros promocionais (R$200)')
print(f'      Os outros {CAPACIDADE - booking_limits[0]} assentos ficam protegidos')
print(f'      para intermediários e última hora.')

# Visualização dos booking limits
fig, ax = plt.subplots(figsize=(10, 3.2))

largura = 0.6
posicoes = [0]
acumulado = 0

segmentos = [
    ('Promocional\n(R$ 200)', booking_limits[0], C['promo']),
    ('Intermediária\n(R$ 500)', int(round(protecoes[0])) - int(round(protecoes[1])) if len(protecoes)>1 else int(round(protecoes[0])), C['inter']),
    ('Última hora\n(R$ 2.000)', int(round(protecoes[1])) if len(protecoes)>1 else 0, C['ultra']),
]

left = 0
for nome, qtd, cor in segmentos:
    ax.barh(0, qtd, left=left, color=cor, alpha=0.85, height=largura, edgecolor='#0d1117', linewidth=1.5)
    if qtd > 8:
        ax.text(left + qtd/2, 0, f'{nome}\n{qtd} assentos', ha='center', va='center',
                fontsize=9, color='#0d1117', fontweight='bold')
    left += qtd

ax.set_xlim(0, CAPACIDADE)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Assentos')
ax.set_yticks([])
ax.set_title('4 · Alocação de capacidade pelo EMSR-b (180 assentos)', weight='bold')
ax.xaxis.set_minor_locator(mticker.MultipleLocator(10))
patches = [mpatches.Patch(color=c, label=n.split('\n')[0]) for n, _, c in segmentos]
ax.legend(handles=patches, loc='upper right', facecolor='#161b22', edgecolor='#30363d', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5 · Cenário determinístico: ocupação máxima vs. Revenue Management

Comparamos duas estratégias usando a demanda **média esperada** (sem aleatoriedade):

- **Ingênua:** vende para qualquer classe na ordem de chegada (barata → cara), sem reservas.
- **Revenue Management:** aplica os booking limits calculados pelo EMSR-b.

In [ ]:
titulo_secao(5, 'Cenário determinístico — demanda = média esperada')

def simular_voo(capacidade, precos, demanda, prot=None):
    """
    Simula venda de assentos em ordem crescente de preço.
    prot=None  → estratégia ingênua
    prot=lista → aplica booking limits do EMSR-b
    """
    n        = len(precos)
    restante = capacidade
    vendidos = np.zeros(n, dtype=int)
    for j in range(n):
        if prot is not None and j < n - 1:
            limite = max(0, restante - int(round(prot[j])))
        else:
            limite = restante
        vender      = min(max(0, int(round(demanda[j]))), limite, restante)
        vendidos[j] = vender
        restante   -= vender
    receita  = float(np.dot(vendidos, precos))
    ocupacao = int(vendidos.sum())
    return vendidos, receita, ocupacao


demanda_base = medias_l[:]

v_n, r_n, o_n = simular_voo(CAPACIDADE, precos_l, demanda_base, prot=None)
v_r, r_r, o_r = simular_voo(CAPACIDADE, precos_l, demanda_base, prot=protecoes)

custo_n  = CUSTO_FIXO + CUSTO_MARGINAL * o_n
custo_r  = CUSTO_FIXO + CUSTO_MARGINAL * o_r
lucro_n  = r_n - custo_n
lucro_r  = r_r - custo_r

print()
linhas = [
    ('Métrica', 'Ingênua (ocup. max.)', 'Revenue Management'),
    ('─' * 22, '─' * 22, '─' * 22),
    ('Passageiros', str(o_n), str(o_r)),
]
for cls, vn, vr in zip(classes['classe'], v_n, v_r):
    linhas.append((f'  {cls}', str(vn), str(vr)))
linhas += [
    ('Lugares vazios', str(CAPACIDADE - o_n), str(CAPACIDADE - o_r)),
    ('Ocupação', f'{100*o_n/CAPACIDADE:.1f}%', f'{100*o_r/CAPACIDADE:.1f}%'),
    ('Receita', brl(r_n), brl(r_r)),
    ('Custo', brl(custo_n), brl(custo_r)),
    ('LUCRO', brl(lucro_n), brl(lucro_r)),
]
for m, n_, rm in linhas:
    print(f'  {m:<22}  {n_:>22}  {rm:>22}')
print()
print(f'  >>> RM vende {o_n - o_r} assentos A MENOS, gera {brl(r_r - r_n)} A MAIS')
print(f'      e {brl(lucro_r - lucro_n)} A MAIS de lucro.')

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
estrategias = ['Ingênua\n(ocup. máx.)', 'Revenue\nManagement']
cores_e     = [C['naive'], C['rm']]

for ax, valores, ylabel, titulo, fmt in [
    (axes[0], [o_n, o_r], 'Assentos', 'Ocupação', lambda v: str(v)),
    (axes[1], [r_n, r_r], 'Receita (R$)', 'Receita total', brl),
    (axes[2], [lucro_n, lucro_r], 'Lucro (R$)', 'Lucro', brl),
]:
    bars = ax.bar(estrategias, valores, color=cores_e, alpha=0.85, width=0.5)
    if titulo == 'Lucro':
        ax.axhline(0, color='white', lw=1.3)
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, weight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x) if isinstance(x, float) and abs(x) > 100 else str(int(x)) if abs(x) < 300 else brl(x)))
    for b, v in zip(bars, valores):
        yp = v + abs(max(valores)) * 0.02
        va = 'bottom'
        if titulo == 'Lucro' and v < 0:
            yp = v - abs(max(valores)) * 0.04
            va = 'top'
        ax.text(b.get_x()+b.get_width()/2, yp, fmt(v), ha='center', va=va, fontsize=9)

fig.suptitle('5 · Avião com menos passageiros pode gerar MAIS lucro', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Simulação de Monte Carlo — 5.000 voos com demanda incerta

O cenário anterior usou a demanda **média** — situação ideal que nunca acontece na prática.
Voos reais têm variabilidade: às vezes a demanda é alta, às vezes baixa.

Rodamos **5.000 simulações**, sorteando demanda com distribuição normal para cada classe.
Os booking limits do EMSR-b são calculados **uma vez** (como na prática real) e aplicados em todos
os voos.

In [ ]:
titulo_secao(6, 'Monte Carlo — 5.000 voos com demanda incerta')

N_SIM  = 5_000
medias = np.array(medias_l)
dps    = np.array(dps_l)

resultados = []
rng = np.random.default_rng(42)
for _ in range(N_SIM):
    d_sim = np.maximum(0, rng.normal(medias, dps))
    _, r_naive, o_naive = simular_voo(CAPACIDADE, precos_l, d_sim, prot=None)
    _, r_rm,    o_rm    = simular_voo(CAPACIDADE, precos_l, d_sim, prot=protecoes)
    resultados.append({'r_naive': r_naive, 'o_naive': o_naive,
                       'r_rm': r_rm,       'o_rm': o_rm})

df = pd.DataFrame(resultados)
df['custo_naive'] = CUSTO_FIXO + CUSTO_MARGINAL * df['o_naive']
df['custo_rm']    = CUSTO_FIXO + CUSTO_MARGINAL * df['o_rm']
df['lucro_naive'] = df['r_naive'] - df['custo_naive']
df['lucro_rm']    = df['r_rm']    - df['custo_rm']

# ── Tabela resumo ─────────────────────────────────────────────────────────────
stats = {
    'Ocupação média (pax)'       : (df['o_naive'].mean(), df['o_rm'].mean()),
    'Receita média'              : (df['r_naive'].mean(), df['r_rm'].mean()),
    'Lucro médio'                : (df['lucro_naive'].mean(), df['lucro_rm'].mean()),
    'Desvio-padrão do lucro'     : (df['lucro_naive'].std(), df['lucro_rm'].std()),
    'Percentil  5% (lucro)'      : (df['lucro_naive'].quantile(.05), df['lucro_rm'].quantile(.05)),
    'Percentil 95% (lucro)'      : (df['lucro_naive'].quantile(.95), df['lucro_rm'].quantile(.95)),
    'Prob. de prejuízo'          : ((df['lucro_naive']<0).mean(), (df['lucro_rm']<0).mean()),
    'Voos lucrativos'            : ((df['lucro_naive']>0).mean(), (df['lucro_rm']>0).mean()),
}

print()
print(f'  {"Métrica":<32}  {"Ingênua":>18}  {"Revenue Mgmt":>18}')
print(f'  {"─"*32}  {"─"*18}  {"─"*18}')
for metrica, (v_n_, v_r_) in stats.items():
    if 'Receita' in metrica or 'Lucro' in metrica or 'Percentil' in metrica or 'Desvio' in metrica:
        fmt = brl
    elif 'Prob' in metrica or 'Voos' in metrica:
        fmt = lambda v: f'{v*100:.1f}%'
    else:
        fmt = lambda v: f'{v:.1f}'
    print(f'  {metrica:<32}  {fmt(v_n_):>18}  {fmt(v_r_):>18}')

print()
lucro_medio_rm   = df['lucro_rm'].mean()
prob_prejuizo_rm = (df['lucro_rm'] < 0).mean()
p05_rm           = df['lucro_rm'].quantile(.05)
p95_rm           = df['lucro_rm'].quantile(.95)

print('─' * 55)
print(f'  Revenue Management · Resultados-chave (5.000 voos):')
print(f'  Lucro médio          {brl(lucro_medio_rm):>15}')
print(f'  Prob. de prejuízo    {prob_prejuizo_rm*100:>14.0f}%')
print(f'  Percentil  5%        {brl(p05_rm):>15}')
print(f'  Percentil 95%        {brl(p95_rm):>15}')
print('─' * 55)

In [ ]:
# ── Gráficos Monte Carlo ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histograma de receita
ax = axes[0, 0]
ax.hist(df['r_naive'], bins=60, alpha=0.6, color=C['naive'], label='Ingênua', density=True)
ax.hist(df['r_rm'],    bins=60, alpha=0.6, color=C['rm'],    label='Revenue Mgmt', density=True)
ax.axvline(df['r_naive'].mean(), color=C['naive'], ls='--', lw=1.5)
ax.axvline(df['r_rm'].mean(),    color=C['rm'],    ls='--', lw=1.5)
ax.set_xlabel('Receita por voo (R$)')
ax.set_ylabel('Densidade')
ax.set_title('Distribuição da Receita', weight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)

# 2. Histograma de lucro
ax2 = axes[0, 1]
ax2.hist(df['lucro_naive'], bins=60, alpha=0.6, color=C['naive'], label='Ingênua', density=True)
ax2.hist(df['lucro_rm'],    bins=60, alpha=0.6, color=C['rm'],    label='Revenue Mgmt', density=True)
ax2.axvline(0, color='white', lw=2, label='Breakeven (lucro = 0)')
ax2.axvline(df['lucro_naive'].mean(), color=C['naive'], ls='--', lw=1.5)
ax2.axvline(df['lucro_rm'].mean(),    color=C['rm'],    ls='--', lw=1.5)
ax2.set_xlabel('Lucro por voo (R$)')
ax2.set_ylabel('Densidade')
ax2.set_title('Distribuição do Lucro', weight='bold')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax2.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)

# 3. CDF do lucro
ax3 = axes[1, 0]
for serie, cor, label in [
    (df['lucro_naive'], C['naive'], 'Ingênua'),
    (df['lucro_rm'],    C['rm'],    'Revenue Mgmt'),
]:
    sorted_vals = np.sort(serie)
    cdf = np.arange(1, len(sorted_vals)+1) / len(sorted_vals)
    ax3.plot(sorted_vals, cdf * 100, color=cor, lw=2, label=label)

ax3.axvline(0, color='white', lw=1.5, ls='--')
ax3.axhline((df['lucro_rm'] < 0).mean() * 100, color=C['rm'], lw=0.8, ls=':')
ax3.set_xlabel('Lucro por voo (R$)')
ax3.set_ylabel('Percentual de voos (%)')
ax3.set_title('CDF do Lucro — risco de prejuízo', weight='bold')
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax3.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)

# 4. Receita por classe — composição média
ax4 = axes[1, 1]
classes_nomes = ['Promo\n(R$200)', 'Inter.\n(R$500)', 'Últ. hora\n(R$2k)']
cores_cl = [C['promo'], C['inter'], C['ultra']]
x_pos = np.arange(len(classes_nomes))
width = 0.35

# Receita esperada por classe (determinística para clareza)
rec_naive = [v * p for v, p in zip(v_n, precos_l)]
rec_rm    = [v * p for v, p in zip(v_r, precos_l)]

bars_n = ax4.bar(x_pos - width/2, rec_naive, width, label='Ingênua',       color=C['naive'], alpha=0.8)
bars_r = ax4.bar(x_pos + width/2, rec_rm,    width, label='Revenue Mgmt', color=C['rm'],    alpha=0.8)
ax4.set_xticks(x_pos); ax4.set_xticklabels(classes_nomes)
ax4.set_ylabel('Receita da classe (R$)')
ax4.set_title('Receita por classe — composição', weight='bold')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax4.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=9)
for b, v in list(zip(bars_n, rec_naive)) + list(zip(bars_r, rec_rm)):
    ax4.text(b.get_x()+b.get_width()/2, v+400, brl(v), ha='center', va='bottom', fontsize=7.5)

fig.suptitle(
    f'6 · Monte Carlo — 5.000 voos  |  '
    f'Lucro médio RM: {brl(df["lucro_rm"].mean())}  |  '
    f'Prob. prejuízo: {(df["lucro_rm"]<0).mean()*100:.0f}%',
    fontsize=12, weight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Box-plot + percentis ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

data_box = [df['lucro_naive'].values, df['lucro_rm'].values]
bp = ax.boxplot(
    data_box,
    labels=['Ingênua', 'Revenue Management'],
    patch_artist=True,
    medianprops  = dict(color='white', linewidth=2),
    whiskerprops = dict(color=C['neutro']),
    capprops     = dict(color=C['neutro']),
    flierprops   = dict(marker='.', alpha=0.3, markersize=2),
    widths=0.45,
)
for patch, cor in zip(bp['boxes'], [C['naive'], C['rm']]):
    patch.set_facecolor(cor)
    patch.set_alpha(0.6)

ax.axhline(0, color='white', lw=1.5, ls='--', label='Breakeven')
ax.set_ylabel('Lucro por voo (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.set_title(
    f'Dispersão do lucro — 5.000 voos\n'
    f'RM: mediana = {brl(df["lucro_rm"].median())}  |  '
    f'P5 = {brl(df["lucro_rm"].quantile(.05))}  |  '
    f'P95 = {brl(df["lucro_rm"].quantile(.95))}',
    weight='bold'
)
ax.legend(facecolor='#161b22', edgecolor='#30363d')
plt.tight_layout()
plt.show()

In [ ]:
# ── Lucro acumulado ao longo de 500 voos (primeiros 500 da simulação) ─────────
n_serie = 500
fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(np.cumsum(df['lucro_naive'][:n_serie]) / 1e6, color=C['naive'], lw=1.8,
        label='Ingênua (ocup. máx.)', alpha=0.9)
ax.plot(np.cumsum(df['lucro_rm'][:n_serie]) / 1e6,    color=C['rm'],    lw=1.8,
        label='Revenue Management',   alpha=0.9)
ax.axhline(0, color='white', lw=1, ls='--')
ax.set_xlabel(f'Número de voos (primeiros {n_serie} da simulação)')
ax.set_ylabel('Lucro acumulado (R$ milhões)')
ax.set_title('Lucro acumulado ao longo dos voos — RM abre vantagem crescente', weight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:.1f}M'))
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.set_facecolor('#161b22')
plt.tight_layout()
plt.show()

---
## 7 · O que a passagem de R$ 99 realmente custa à empresa?

Se o EMSR-b já define quantos assentos vender em cada classe, de onde vem o R\$ 99 que
aparece nos anúncios e no Google Flights?

Na prática, é comum a empresa reservar **1–2 assentos por voo fora da otimização**
como **isca de marketing**: aparecer em buscas com "a partir de R\$ 99", ranquear em
comparadores de preço e gerar brand awareness. O custo de oportunidade é mínimo.

In [ ]:
titulo_secao(7, 'Custo de oportunidade da passagem-isca de R$ 99')

PRECO_ISCA = 99
print()
print(f'  Receita de referência (Revenue Management): {brl(r_r)}')
print()
print(f'  {"Assentos-isca":<18}  {"Perda de receita":<22}  {"% da receita total":<22}  {"Receita c/ isca"}')
print(f'  {"─"*18}  {"─"*22}  {"─"*22}  {"─"*18}')
for n_isca in [1, 2, 3, 5]:
    perda            = n_isca * (precos_l[0] - PRECO_ISCA)
    receita_com_isca = r_r - perda
    impacto_pct      = 100 * perda / r_r
    print(f'  {n_isca:<18}  {brl(perda):<22}  {impacto_pct:<21.2f}%  {brl(receita_com_isca)}')

print()
print('  >>> CONCLUSÃO: o custo de 1–2 assentos a R$99 representa < 0,1% da receita do voo.')
print('      O ganho em visibilidade (SEO, comparadores) supera amplamente esse custo.')
print('      O motor de receita real são as outras ~178 cadeiras.')

# Gráfico de sensibilidade
fig, ax = plt.subplots(figsize=(10, 4))
n_iscas   = np.arange(0, 21)
perdas    = n_iscas * (precos_l[0] - PRECO_ISCA)
receitas  = r_r - perdas

ax.fill_between(n_iscas, receitas / 1000, r_r / 1000, alpha=0.25, color=C['naive'], label='Perda de receita')
ax.plot(n_iscas, receitas / 1000, color=C['rm'], lw=2.5, label='Receita com N iscas')
ax.axhline(r_r / 1000, color=C['neutro'], lw=1, ls=':', label=f'Sem isca: {brl(r_r)}')
ax.set_xlabel('Número de assentos-isca vendidos a R$ 99')
ax.set_ylabel('Receita total (R$ mil)')
ax.set_title('7 · Sensibilidade da receita ao número de assentos-isca', weight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:.0f}k'))
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.set_facecolor('#161b22')
plt.tight_layout()
plt.show()

---
## 8 · Contexto histórico e conclusões

### A principal descoberta

| Estratégia | Pax | Ocupação | Resultado |
|---|---|---|---|
| Lotar o avião (ingênua) | 180 | 100% | ❌ Pode dar **prejuízo** |
| Revenue Management | ~163 | ~90,6% | ✅ Gera **lucro** |

**Um avião cheio não é necessariamente um avião lucrativo.**

---

### Contexto histórico real

O Revenue Management moderno nasceu na **American Airlines em 1985**, em resposta direta
à entrada da **People Express** — startup de baixíssimo custo que ameaçava destruir as
margens das companhias tradicionais após a desregulamentação americana de 1978.

Em vez de igualar os preços da concorrente em **todos** os assentos, o CEO **Robert Crandall**
desenvolveu o **DINAMO** (*Dynamic Inventory Optimization and Maintenance Optimizer*):
tarifas promocionais controladas por **quantidade**, **antecedência** e **não-reembolso**.

A equipe recebeu o **Prêmio Edelman de 1991** (o mais importante de Pesquisa Operacional),
atribuindo ao sistema mais de **US\$ 1,4 bilhão** em receita incremental em três anos.
A People Express encerrou as atividades em 1987.

### Base matemática

| Contribuição | Autor | Ano | Instituição |
|---|---|---|---|
| Regra de proteção de assentos (2 classes) | Ken Littlewood | 1972 | BOAC / British Airways |
| Heurística EMSR-a e EMSR-b (n classes) | Peter Belobaba | 1987–1989 | MIT |
| Sistema DINAMO (implementação industrial) | American Airlines | 1985 | — |

In [ ]:
titulo_secao(8, 'Resumo final — Revenue Management em números')

print()
print('  ╔══════════════════════════════════════════════════════════════════╗')
print('  ║          REVENUE MANAGEMENT · RESUMO DO NOTEBOOK                ║')
print('  ╠══════════════════════════════════════════════════════════════════╣')
print(f'  ║  Avião: Airbus A320 / 180 assentos / voo de 2h30               ║')
print(f'  ║  Custo fixo: {brl(CUSTO_FIXO):<20} Custo marginal: {brl(CUSTO_MARGINAL):<12} ║')
print('  ╠══════════════════════════════════════════════════════════════════╣')
print(f'  ║  Cenário determinístico (demanda = média)                       ║')
print(f'  ║    Ingênua   → {o_n:>3} pax · {brl(r_n):>12} receita · {brl(lucro_n):>12} lucro ║')
print(f'  ║    Rev. Mgmt → {o_r:>3} pax · {brl(r_r):>12} receita · {brl(lucro_r):>12} lucro ║')
print('  ╠══════════════════════════════════════════════════════════════════╣')
print(f'  ║  Monte Carlo (5.000 voos, demanda aleatória)                    ║')
print(f'  ║    Lucro médio RM       {brl(df["lucro_rm"].mean()):>15}                     ║')
print(f'  ║    Prob. de prejuízo    {(df["lucro_rm"]<0).mean()*100:>14.0f}%                     ║')
print(f'  ║    Percentil  5%        {brl(df["lucro_rm"].quantile(.05)):>15}                     ║')
print(f'  ║    Percentil 95%        {brl(df["lucro_rm"].quantile(.95)):>15}                     ║')
print('  ╠══════════════════════════════════════════════════════════════════╣')
print('  ║  Lição central:                                                  ║')
print('  ║  "A pergunta não é como lotar o avião —                          ║')
print('  ║   é como extrair o máximo valor de cada lugar disponível."       ║')
print('  ╚══════════════════════════════════════════════════════════════════╝')

# Painel visual final
fig, ax = plt.subplots(figsize=(13, 5))

# Scatter: ocupação × lucro nos 5.000 voos
sample = df.sample(1000, random_state=42)
ax.scatter(sample['o_naive'], sample['lucro_naive'] / 1000,
           color=C['naive'], alpha=0.25, s=12, label='Ingênua')
ax.scatter(sample['o_rm'],    sample['lucro_rm']    / 1000,
           color=C['rm'],    alpha=0.25, s=12, label='Revenue Management')
ax.axhline(0, color='white', lw=1.5, ls='--')
ax.set_xlabel('Ocupação (passageiros)')
ax.set_ylabel('Lucro por voo (R$ mil)')
ax.set_title(
    '8 · Ocupação × Lucro — 1.000 voos amostrados (de 5.000 simulados)\n'
    'RM = menor ocupação, maior e mais estável lucro',
    weight='bold'
)
ax.legend(facecolor='#161b22', edgecolor='#30363d', markerscale=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:.0f}k'))
ax.set_facecolor('#161b22')
plt.tight_layout()
plt.show()

print()
print('✅  Notebook concluído.')
print('    Notebook didático/editorial — @financelabr')
print('    Parâmetros de demanda são hipotéticos para fins educacionais.')